# OpenCodeInstruct → High-Quality 10K subset (Qwen ChatML)

Lọc `nvidia/OpenCodeInstruct` (5M mẫu) còn ~10K mẫu chất lượng cao để fine-tune Qwen2.5-Coder:

1. `average_test_score >= 0.8`
2. 3 điểm LLM-judge (`requirement_conformance`, `logical_correctness`, `edge_case_consideration`) đều `>= 4`
3. Lấy 10K mẫu, format sang ChatML (`<|im_start|>...`), lưu ra đĩa.

**Lưu ý schema thật của dataset** (đã kiểm tra): các cột là `input`, `output`, `llm_judgement`, `average_test_score`, ... và **cả `llm_judgement` lẫn `average_test_score` đều là *string*** (vd `"0.9"`, `'{"requirement_conformance": {"score": 5, ...}}'`) → phải ép kiểu trước khi so sánh.

**Chạy thử nhanh:** để `TARGET_N = 200` ở Config → xong trong ~1 phút. Chạy thật thì đổi thành `10_000`.

In [1]:
!pip install -q datasets

## 1. Config

In [2]:
import json
import statistics
from pathlib import Path
from datasets import load_dataset, Dataset

DATASET_ID = "nvidia/OpenCodeInstruct"

TARGET_N        = 10_000       
MIN_TEST_SCORE  = 0.8
MIN_JUDGE_SCORE = 4
SEED            = 42

# "stream": duyệt streaming, dừng khi đủ TARGET_N (nhanh, ít RAM, không cần tải 5M mẫu)
# "full"  : tải toàn bộ rồi ds.filter(num_proc=...) (chậm, tốn đĩa/RAM, nhưng random trên toàn bộ pool)
MODE = "stream"
# Chỉ dùng cho MODE="stream": shuffle trong buffer để mẫu không bị lệch theo thứ tự đầu dataset
SHUFFLE_BUFFER = 10_000
# Chỉ dùng cho MODE="full"
NUM_PROC = 4

INSTRUCTION_FIELD = "input"
RESPONSE_FIELD    = "output"

# True: thêm system prompt mặc định của Qwen vào đầu mỗi sample
ADD_SYSTEM_PROMPT = False
SYSTEM_PROMPT = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."

OUT_DIR = Path("./opencode_hq")
OUT_DIR.mkdir(exist_ok=True)

/home/aioz/Aioz-Hoang/code-generation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Filter

In [3]:
JUDGE_KEYS = ("requirement_conformance", "logical_correctness", "edge_case_consideration")


def _to_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def is_high_quality(ex):
    # Filter 1: average_test_score (string "0.9" -> 0.9)
    test_score = _to_float(ex.get("average_test_score"))
    if test_score is None or test_score < MIN_TEST_SCORE:
        return False

    # Filter 2: tất cả LLM judge scores >= MIN_JUDGE_SCORE
    judgement = ex.get("llm_judgement")
    if isinstance(judgement, str):
        try:
            judgement = json.loads(judgement)
        except (json.JSONDecodeError, TypeError):
            return False
    if not isinstance(judgement, dict):
        return False

    for key in JUDGE_KEYS:
        entry = judgement.get(key)
        score = _to_float(entry.get("score") if isinstance(entry, dict) else entry)
        if score is None or score < MIN_JUDGE_SCORE:
            return False

    # Bỏ mẫu rỗng
    if not (ex.get(INSTRUCTION_FIELD) or "").strip() or not (ex.get(RESPONSE_FIELD) or "").strip():
        return False
    return True


head = list(load_dataset(DATASET_ID, split="train", streaming=True).take(50))
n_ok = sum(is_high_quality(e) for e in head)
print(f"{n_ok}/{len(head)} mẫu đầu tiên pass filter")

31/50 mẫu đầu tiên pass filter


## 3. Run filter

In [4]:
import time
t0 = time.time()

if MODE == "stream":
    stream = load_dataset(DATASET_ID, split="train", streaming=True)
    stream = stream.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)

    collected, scanned = [], 0
    for ex in stream:
        scanned += 1
        if is_high_quality(ex):
            collected.append(ex)
            if len(collected) >= TARGET_N:
                break
        if scanned % 5000 == 0:
            print(f"  scanned {scanned:,} | collected {len(collected):,}")
    final = Dataset.from_list(collected)
    print(f"Scanned {scanned:,} mẫu, pass rate ~{len(collected)/scanned:.1%}")

elif MODE == "full":
    ds = load_dataset(DATASET_ID, split="train")
    filtered = ds.filter(is_high_quality, num_proc=NUM_PROC)
    print(f"High-quality pool: {len(filtered):,} / {len(ds):,}")
    final = filtered.shuffle(seed=SEED)
    if len(final) >= TARGET_N:
        final = final.select(range(TARGET_N))
    else:
        print(f"WARNING: chỉ có {len(final)} mẫu — lấy hết")
else:
    raise ValueError("MODE phải là 'stream' hoặc 'full'")

print(f"Final: {len(final):,} samples ({time.time()-t0:.0f}s)")
if len(final) < TARGET_N:
    print(f"WARNING: mới thu được {len(final)}/{TARGET_N}")

  scanned 5,000 | collected 1,744
  scanned 10,000 | collected 3,541
  scanned 15,000 | collected 5,426
  scanned 20,000 | collected 7,237
  scanned 25,000 | collected 8,874
Scanned 28,277 mẫu, pass rate ~35.4%
Final: 10,000 samples (41s)


## 4. Statistics

In [5]:
from collections import Counter

scores = [float(s) for s in final["average_test_score"]]
print(f"Test score — min: {min(scores):.2f}, mean: {statistics.mean(scores):.2f}, max: {max(scores):.2f}")

judge = {k: [] for k in JUDGE_KEYS}
for j in final["llm_judgement"]:
    j = json.loads(j) if isinstance(j, str) else j
    for k in JUDGE_KEYS:
        judge[k].append(float(j[k]["score"] if isinstance(j[k], dict) else j[k]))
for k, v in judge.items():
    print(f"{k:26s} mean {statistics.mean(v):.2f}  min {min(v):.0f}")

print("domain:", dict(Counter(final["domain"])))
print("generation_algorithm:", dict(Counter(final["generation_algorithm"])))

Test score — min: 0.80, mean: 0.95, max: 1.00
requirement_conformance    mean 4.98  min 4
logical_correctness        mean 4.98  min 4
edge_case_consideration    mean 4.37  min 4
domain: {'generic': 6515, 'algorithmic': 3485}
generation_algorithm: {'self-instruct': 8923, 'evol-instruct': 1077}


## 5. Save + format to Qwen ChatML

In [6]:
final.save_to_disk(str(OUT_DIR / "raw"))

def format_qwen(ex):
    sys_part = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n" if ADD_SYSTEM_PROMPT else ""
    return {
        "text": (
            sys_part
            + f"<|im_start|>user\n{ex[INSTRUCTION_FIELD].strip()}<|im_end|>\n"
            + f"<|im_start|>assistant\n{ex[RESPONSE_FIELD].strip()}<|im_end|>"
        )
    }

formatted = final.map(format_qwen, remove_columns=final.column_names)
formatted.save_to_disk(str(OUT_DIR / "chatml"))
formatted.to_json(str(OUT_DIR / "chatml.jsonl")) 

print(f"Saved {len(formatted):,} samples -> {OUT_DIR.resolve()}")
print("-" * 60)
print(formatted[0]["text"][:800])

Creating json from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 147.65ba/s]

Saved 10,000 samples -> /home/aioz/Aioz-Hoang/code-generation/opencode_hq
------------------------------------------------------------
<|im_start|>user
You are given a string `s` consisting of '(' and ')' characters. Your task is to determine the length of the longest valid (well-formed) parentheses substring. A valid parentheses substring is defined as a substring that has an equal number of opening and closing parentheses, with each opening parenthesis having a corresponding closing parenthesis in the correct order. Write a function `longest_valid_parentheses(s)` that takes a string `s` as input and returns the length of the longest valid parentheses substring.

**Sample Input:**
```
(()())()
```

**Sample Output:**
```
8
```<|im_end|>
<|im_start|>assistant
```python
def longest_valid_parentheses(s: str) -> int:
    """
    Returns the length of the longest valid (well-formed) parentheses substring in the given string `
